# RAG

In [3]:
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

openai_client = OpenAI()
documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo check that the local server is running, you can also run:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'The FAQ context doesn’t mention **Olama** specifically, but it does say you can run the course locally if you’re comfortable setting up the needed tools.\n\nSo, based on the FAQ:\n\n- You can run the course locally instead of using Codespaces.\n- You’ll need to set up the required tools yourself, such as **Python, `uv`, Jupyter, Docker**, and anything else needed for the module.\n- If you run locally, you should **document your setup** and keep it **reproducible**.\n\nIf you meant **Ollama** specifically, I don’t have any local-run instructions for it in the provided context.'

In [7]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

## Asking without the tools

In [8]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—if enrollment is still open, you can join it.\n\nTo find out for sure:\n1. Check the course page for an enrollment deadline or “join” button.\n2. If it’s through a school, platform, or instructor, contact them directly and ask whether late enrollment is allowed.\n3. If the course has already started, ask whether you can still catch up or join the next session.\n\nIf you want, I can help you draft a short message asking to join.'

## Defining the tool

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Sending the question with the tool

In [11]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll register late start can I join after course started"}', call_id='call_ZJYdfc9CCHVPfJVPPWBv3ixz', name='search', type='function_call', id='fc_0535be5ebf93d084006a395a0f34ac8191abf9da3defe6cdd2', namespace=None, status='completed')]

## Executing the function and sending the result back

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

## Asking the model again 

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nYou don’t need a confirmation email or formal acceptance to start learning; just begin with the course materials and submit homework while the forms/deadlines are open.\n\nIf you want a certificate, make sure you complete the project and submit it while submissions are still being accepted.'

## Token usage and cost of all these

In [15]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(775, 65)

In [16]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Developer prompt

In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## Function-call helper

In [18]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [19]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment admission late join FAQ"}
function_call: search {"query":"new student join the course register enroll discovered course FAQ"}


## Full agent loop

In [20]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If your goal is to get a certificate, you’ll need to submit your project while the course is still accepting submissions. Also, certificates are only available if you finish with a live cohort, not in self-paced mode.

If you want, I can also help you with how to start the course or explain the certificate requirements in more detail.


## Wrapping it in a function

In [21]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [22]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local FAQ"}
iteration #2...
function_call: search {"query":"Ollama run llama3 local server localhost 11434 Python client ollama FAQ"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download and install the `.pkg`
     - **Windows**: download and install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check the local server**
   ```bash
   curl http://localhost:11434
   ```
   If it’s running, you should get a response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       mes

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: download and install the `.pkg`\n     - **Windows**: download and install the `.msi`\n     - **Linux**: run:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat interface.\n\n3. **Check the local server**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s running, you should get a response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, you may need to restart the server with:\n```bash\nolla

In [23]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join"}
function_call: search {"query":"late enrollment course join after start discovered the course"}
function_call: search {"query":"enrollment still open course FAQ join after course started"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

A couple of notes:
- You can start learning and following the materials even if you discovered it late.
- If you want a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help you figure out the best way to catch up quickly or explain the certificate requirements.


'Yes — you can still join the course.\n\nA couple of notes:\n- You can start learning and following the materials even if you discovered it late.\n- If you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out the best way to catch up quickly or explain the certificate requirements.'

## Encouraging multiple searches

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while submissions are still open and finish with the live cohort.

If you'd like, I can also tell you how to start the course or explain the certificate requirements in more detail.


"Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while submissions are still open and finish with the live cohort.\n\nIf you'd like, I can also tell you how to start the course or explain the certificate requirements in more detail."

## Restricting off topic questrions

In [25]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition accept decline white black d4 c4"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to draw Black’s **d-pawn** away from the center, giving White more control over the middle of the board.

A few common ideas:
- White aims for **central control**
- Black can **accept** the gambit by taking the c-pawn, or **decline** it by keeping the pawn structure
- It’s one of the most famous and well-studied openings in chess

If you want, I can also explain:
- the **Queen’s Gambit Accepted**
- the **Queen’s Gambit Declined**
- or the **main idea in simple terms**


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to draw Black’s **d-pawn** away from the center, giving White more control over the middle of the board.\n\nA few common ideas:\n- White aims for **central control**\n- Black can **accept** the gambit by taking the c-pawn, or **decline** it by keeping the pawn structure\n- It’s one of the most famous and well-studied openings in chess\n\nIf you want, I can also explain:\n- the **Queen’s Gambit Accepted**\n- the **Queen’s Gambit Declined**\n- or the **main idea in simple terms**'

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
If you mean the chess opening, the **Queen’s Gambit** is a chess opening that starts with the moves **1. d4 d5 2. c4**.

If you meant something else in the course context, I couldn’t find a FAQ entry for “queen gambit.”  
Would you like to explore anything else?


'If you mean the chess opening, the **Queen’s Gambit** is a chess opening that starts with the moves **1. d4 d5 2. c4**.\n\nIf you meant something else in the course context, I couldn’t find a FAQ entry for “queen gambit.”  \nWould you like to explore anything else?'

# Registreing the tool

In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [29]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [30]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [32]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [33]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [34]:
result.cost

CostInfo(input_cost=Decimal('0.00111075'), output_cost=Decimal('0.0011385'), total_cost=Decimal('0.00224925'))

In [35]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally local installation Ollama run locally"}', ca

In [36]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [37]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"FAQ course teaching assistant question about course logistics"}', c